In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

In [3]:
# Load recipe-level data (already enriched with ingredient categories)
recipes_df = pd.read_csv("recipes_df.csv")

# Load external health datasets
life_expectancy_df = pd.read_csv("life-expectancy/life-expectancy.csv")
child_mortality_df = pd.read_csv("child-mortality/child-mortality.csv")
maternal_mortality_df = pd.read_csv("maternal-mortality/maternal-mortality.csv")

# Load WHO health statistics Excel
who_health = pd.read_excel("world_health_statistics_2024.xlsx", sheet_name=None)

# Preview keys (sheet names) and head of the recipe dataset
recipes_df.columns, list(who_health.keys()), recipes_df.head(3)


(Index(['url', 'title', 'steps', 'rating', 'comments', 'dish_type', 'cuisine',
        'continent', 'sub_region', 'Calories', 'Fat', 'Carbs', 'Protein',
        'prep_time', 'cook_time', 'additional_time', 'total_time', 'servings',
        'ingredients', 'num_ingredients', 'num_steps', 'log_prep_time',
        'log_cook_time', 'log_additional_time', 'log_total_time',
        'log_Calories', 'log_Fat', 'log_Carbs', 'log_Protein',
        'ingredients_str', 'ingredients_cleaned', 'ingredient_semantics'],
       dtype='object'),
 ['readme', 'data'],
                                                  url             title  \
 0  https://www.allrecipes.com/recipe/19344/homema...  Homemade Lasagna   
 1  https://www.allrecipes.com/recipe/223042/chick...  Chicken Parmesan   
 2  https://www.allrecipes.com/recipe/8887/chicken...   Chicken Marsala   
 
                                                steps  rating comments  \
 0  ['Gather all ingredients.Dotdash Meredith Food...     4.6       [] 

In [4]:
# Load the WHO health indicators sheet
who_df = who_health["data"]

# Preview unique indicators available
indicators_summary = who_df["IND_NAME"].value_counts().head(30)

# Also preview the structure
who_df.head(3), indicators_summary


(                                            IND_NAME DIM_GEO_NAME  \
 0             Adolescent birth rate (per 1000 women)  Afghanistan   
 1             Adolescent birth rate (per 1000 women)  Afghanistan   
 2  Age-standardized mortality rate attributed to ...  Afghanistan   
 
          IND_CODE DIM_GEO_CODE  DIM_TIME_YEAR           DIM_1_CODE  \
 0  MDG_0000000003          AFG           2021  AGEGROUP_YEARS15-19   
 1  MDG_0000000003          AFG           2021  AGEGROUP_YEARS10-14   
 2      SDGAIRBODA          AFG           2019             SEX_BTSX   
 
    VALUE_NUMERIC VALUE_STRING  \
 0       62.00000         62.0   
 1       18.00000         18.0   
 2      265.66452        265.7   
 
                                       VALUE_COMMENTS  
 0  Afghanistan 2022-2023 Multiple Indicator Clust...  
 1  Afghanistan 2022-2023 Multiple Indicator Clust...  
 2                                                NaN  ,
 IND_NAME
 Life expectancy at birth (years)                          

In [5]:
# Select the 6 key indicators by substring filtering
health_indicators = [
    "Life expectancy at birth (years)",
    "Healthy life expectancy at birth (years)",
    "Prevalence of obesity among adults",
    "Prevalence of hypertension among adults",
    "Total alcohol per capita",
    "UHC: Service coverage index"
]

# Filter the WHO dataset for those indicators
filtered_health_df = who_df[
    who_df["IND_NAME"].isin(health_indicators)
].copy()

# Get latest available year per country/indicator
filtered_health_df.sort_values("DIM_TIME_YEAR", ascending=False, inplace=True)
filtered_health_df = filtered_health_df.drop_duplicates(subset=["IND_NAME", "DIM_GEO_NAME"])

# Pivot to wide format
health_wide = filtered_health_df.pivot(
    index="DIM_GEO_NAME", columns="IND_NAME", values="VALUE_NUMERIC"
).reset_index().rename(columns={"DIM_GEO_NAME": "cuisine"})

health_wide.head()


IND_NAME,cuisine,Healthy life expectancy at birth (years),Life expectancy at birth (years),UHC: Service coverage index
0,Afghanistan,51.312912,59.126904,40.884609
1,African Region,54.617283,63.552441,44.389660
2,Albania,67.758415,78.612389,63.768951
3,Algeria,66.375206,75.977654,74.111641
4,Andorra,NaN,NaN,78.862480


In [6]:
# Safely evaluate ingredient semantics if needed
recipes_df["ingredient_semantics"] = recipes_df["ingredient_semantics"].apply(eval)

# Recompute luxury score from categories: dairy, seafood, nut, sweetener
luxury_keywords = ["dairy", "seafood", "nut", "sweetener"]

def count_luxury(sem_list):
    return sum(1 for item in sem_list if item in luxury_keywords)

recipes_df["luxury_score"] = recipes_df["ingredient_semantics"].apply(count_luxury)

# Now re-aggregate
cuisine_health_features = recipes_df.groupby("cuisine").agg({
    "Calories": "mean",
    "Fat": "mean",
    "Carbs": "mean",
    "Protein": "mean",
    "num_ingredients": "mean",
    "luxury_score": "mean",
    "sub_region": "first",
    "continent": "first"
}).reset_index()

# Merge with health data
merged_health_df = cuisine_health_features.merge(health_wide, on="cuisine", how="left")

merged_health_df.head()


,cuisine,Calories,Fat,Carbs,Protein,num_ingredients,luxury_score,sub_region,continent,Healthy life expectancy at birth (years),Life expectancy at birth (years),UHC: Service coverage index
0,Afghanistan,454.600000,14.400000,63.800000,19.000000,10.000000,0.800000,Southern Asia,Asia,51.312912,59.126904,40.884609
1,Algeria,303.555556,13.222222,34.444444,12.333333,14.000000,0.222222,Northern Africa,Africa,66.375206,75.977654,74.111641
2,Argentina,309.896552,18.900000,26.633333,8.833333,8.733333,0.233333,South America,Americas,64.794365,77.577904,78.525368
3,Armenia,348.812500,17.000000,36.437500,13.250000,7.375000,0.375000,Western Asia,Asia,64.013306,68.598511,68.193108
4,Australia,312.823529,15.055556,32.166667,12.555556,12.944444,1.833333,Australia and New Zealand,Oceania,70.606361,84.897820,86.778778
